# 全量审计：正常／幻觉、任意样本与 head

只读取本次保存的文件，不加载模型权重。Layer/head 从 0 开始。红色为幻觉，特殊 token 排除。

修改下面的任务、正负回答类别、样本 ID、layer/head，再运行相应单元。完整来源图由实际保存的 Q/K 重建；不同设备的浮点内核可能有小差异，原生 history 矩阵可以核对。


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
from IPython.display import display, Image, HTML

# 若 notebook 放在仓库外，只需要修改 PROJECT 和 OUTPUT。
PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'research_dataset.py').exists()),
               Path('/share/home/tm902089733300000/a903202310/lys/research/graph'))
OUTPUT = Path.cwd() if (Path.cwd() / 'review_index.json').exists() else PROJECT / 'experiments/reanchor_flow/outputs/attention_audit_v3'
sys.path.insert(0, str(PROJECT))
from experiments.reanchor_flow.attention_audit_plot import plot_sample, plot_cohort, plot_onset, source_unit_paths
from experiments.reanchor_flow.attention_audit import reconstruct_attention
records = json.loads((OUTPUT / 'review_index.json').read_text())
print('Samples:', len(records), 'output:', OUTPUT)


In [ ]:
SPLIT, TASK, CASE = 'test', 'QA', 'positive'   # CASE: positive / negative / unknown
choices = [e for e in records if e['split'] == SPLIT and e['task_type'] == TASK and e['answer_class'] == CASE]
print([(e['sample_id'], e['normal_tokens'], e['hallucinated_tokens']) for e in choices[:30]])
SAMPLE_ID = choices[0]['sample_id'] if choices else None  # 可修改为任意实际 ID
LAYER, HEAD = 0, 0
entry = next(e for e in records if e['split'] == SPLIT and e['task_type'] == TASK and e['sample_id'] == SAMPLE_ID)
path = OUTPUT / entry['path']
print(entry)


## 完整标签文本与任意 head

图左为预测位置读取，中间为载体复用，右为完整来源矩阵。红色区和红色边条表示幻觉 token。所有 head 的原始数据都在，示例选了哪些 head 不影响总体统计。


In [ ]:
display(HTML((OUTPUT / entry['text']).read_text()))
display(Image(filename=str(plot_sample(path, layer=LAYER, head=HEAD, title=f'{SPLIT}/{TASK}/{SAMPLE_ID}'))))


## 任意指标的整体 N / H / 配对差异

只在 train 上选取要进一步验证的 head，再看 test 中同一个 head。下面可以选择全部已保存指标，不限于默认总览中的六项。


In [ ]:
cohort = OUTPUT / 'cohorts' / f'{SPLIT}_{TASK}.npz'
with np.load(cohort) as z:
    metric_names = z['metric_names'].tolist()
print(metric_names)
METRIC = 'evidence_share'
figure = plot_cohort(cohort, metric_names=[METRIC])
display(Image(filename=str(figure)))
with np.load(cohort) as z:
    k = metric_names.index(METRIC)
    print({'head': (LAYER, HEAD), 'N': float(z['raw_mean'][0,k,LAYER,HEAD]),
           'H': float(z['raw_mean'][1,k,LAYER,HEAD]),
           'matched_H_minus_N': float(z['matched_mean'][k,LAYER,HEAD]),
           'source_count': int(z['matched_sources'][k,LAYER,HEAD]),
           'CI95': z['matched_ci95'][:,k,LAYER,HEAD].tolist(),
           'BY_q': float(z['matched_q_by'][k,LAYER,HEAD])})


## 幻觉起点附近：完整正常对照窗口

不把句子边界当作内部节点。这里使用标签中的幻觉起点，事后检查其前后结构；正常窗口在同一回答内按位置、token 类型匹配。没有合格对照时结果留空，不补零。


In [ ]:
display(Image(filename=str(plot_onset(cohort, METRIC, LAYER, HEAD))))


## 两跳来源单位：同一个具体 writer / reader

下列量是 `reader 对载体的 attention × writer 在载体对某来源单位的 attention`，保留真实层序与具体来源单位。它检查来源连接结构，不等于证明该事实内容经过了此路径。


In [ ]:
WRITER = (0, 0)
READER = (1, 0)  # reader 层必须严格更深
units = source_unit_paths(path, WRITER, READER)
print('Source units:', units['unit_names'].tolist())
print('Shapes:', units['two_hop'].shape, units['direct'].shape)
# 展示的单位按总路径系数排序；所有单位仍保留在 units 中。
import matplotlib.pyplot as plt
selected = np.argsort(np.nansum(units['two_hop'], axis=0))[-min(8, len(units['unit_names'])):]
if len(selected):
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)
    with np.load(path.with_suffix('.labels.npz')) as z:
        labels = z['labels']
    for ax, field in zip(axes, ['direct', 'two_hop']):
        for i in selected:
            ax.plot(units[field][:, i], label=str(units['unit_names'][i]))
        for a,b in np.flatnonzero(np.diff(np.r_[False, labels == 1, False])).reshape(-1,2):
            ax.axvspan(a-.5, b-.5, color='#ed6480', alpha=.2)
        ax.set(title=field, xlabel='Predicted response token index', ylabel='Structural attention coefficient')
    axes[0].legend(fontsize=8, ncol=2)
    plt.show()


## 核对 Q/K 重建与原生保存值

此检查不加载模型。若改换硬件或精度，小的浮点差异需要如实记录。


In [ ]:
full = reconstruct_attention(path, LAYER, HEAD)
with np.load(path) as z:
    start = int(z['response_start'])
with np.load(path.with_suffix('.history.npz')) as z:
    native = z[f'L{LAYER}'][HEAD]
print('maximum history reconstruction error:', float(np.max(np.abs(full[:, start-1:] - native))))
print('Full source map shape:', full.shape)
